In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import numpy as np
from pathlib import Path

In [ ]:
results_dir = Path(os.environ["EPHEMERAL"]) / "pyspoc_runs/results/concat/"

N = 200
P = 100

dataset_folder = f'gtex_geneSub_N{N}_P{P}__mode_partition'

dm_file = 'datamatrix__3configs__filtered_st85_dt25.parquet'

data_path = results_dir / dataset_folder / dm_file

In [ ]:
data_matrix = pd.read_parquet(data_path)

## OPTIONAL: REMOVE PATHS FROM DATA MATRIX COLUMN NAMES
## (EXTERNAL MODULES GET SAVED WITH THE FULL PATH)

stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

data_matrix_names = data_matrix.columns.str.replace(stats_path, ".", regex=False)
data_matrix.columns = data_matrix_names

print(data_matrix.shape[0],data_matrix.shape[1])

In [ ]:
# data_matrix.loc[:,num_unique_val_per_stat<=1]

In [ ]:
short_stat_names = pd.Series(data_matrix.columns.str.split(".").str[-4:-1].str.join("."), index=data_matrix.columns)

In [ ]:
print('the following cols. are constant across all datasets:')
num_unique_val_per_stat = data_matrix.nunique()
print(num_unique_val_per_stat[num_unique_val_per_stat <= 1])

# remove constant columns
print('removing constant cols')
df_processed = data_matrix.loc[:,~(num_unique_val_per_stat <= 1)].copy()

In [ ]:
T = df_processed.T
dups = T[T.duplicated(keep=False)] # all columns that have a copy
groups = [[len(list(g.index))] + list(g.index) for _, g in dups.groupby(list(dups.columns), dropna=False)]
groups_short = [[len(list(g.index))] + list(short_stat_names[g.index]) for _, g in dups.groupby(list(dups.columns), dropna=False)]

## ---- FASTER WAY TO FIND DUPS, HASH THE ENTIRE COL
# h = df_processed.apply(lambda c: hash(c.to_numpy().tobytes())) # one hash per column
# groups = [list(cols) for cols in h.groupby(h).groups.values() if len(cols) > 1]
# for g in groups[:10]:
#     print(g)

print(len(groups), "groups of identical columns")
df_dup_cols = pd.DataFrame(groups_short)

In [ ]:
df_dup_cols.sort_values(by=0,ascending=True).drop_duplicates()

In [ ]:
print("NaN:", np.isnan(df_processed).sum().sum(), " inf:", np.isinf(df_processed).sum().sum(), " max |x|:", np.nanmax(np.abs(df_processed)))

In [ ]:
df_dedup = df_processed.T.drop_duplicates().T.copy()

# CURRENTLY I MANUALLY MAKE THE DETERMINNANT LOG
# TO BE IMPLEMENTED IN LIBRARY
detCols = [v for v in df_dedup.columns if 'pyspoc.reducers.basic.Determinant.scaled' in v]

# clipping above at 1e200 to remove inf
logdet_per_dim = (np.log(df_dedup[detCols].abs().clip(lower=1e-300,upper=1e200))) / P
sign = np.sign(df_dedup[detCols])
df_dedup[detCols] = logdet_per_dim

In [ ]:
# logdet_per_dim

In [ ]:
df_processed.shape[1]-len(groups) == df_dedup.shape[1]

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
(df_dedup.max(axis=0)-df_dedup.min(axis=0)).describe()

In [ ]:
print("NaN:", np.isnan(df_dedup).sum().sum(), " inf:", np.isinf(df_dedup).sum().sum(), " max |x|:", np.nanmax(np.abs(df_dedup)))

In [ ]:
# # CHECK FOR EXTREMELY BIG VALUES:
# mx = df_dedup.max().sort_values(ascending=False)
# print(mx.head(15))
# big = df_dedup.columns[df_dedup.max() > 1e40]
# print(len(big), "columns above 1e40")
# print(df_dedup[big].idxmax().value_counts().head())

In [ ]:
did_seed_concat = df_dedup.index.str.split('_').str[:-1].str.join(sep='_')
between = df_dedup.groupby(did_seed_concat).mean().var(axis=0)
within = df_dedup.groupby(did_seed_concat).var().mean(axis=0)
rough_icc = (between / (between + within)).sort_values()
rough_icc.plot(kind='hist')

In [ ]:
col_icc_filter = rough_icc[rough_icc>0.25].index

df_icc_filt = df_dedup.loc[:,col_icc_filter]

### clustering the data matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
nb_dir = Path("hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/py_scripts/dataviz_notebooks/")
FIG_DIR = nb_dir.resolve() / 'figures'
os.makedirs(FIG_DIR,exist_ok=True)

In [ ]:
tissue_fine = df_dedup.index.str.split('__').str[1]
tissue_broad = df_dedup.index.str.split('__').str[0]

In [ ]:
# tissue_broad.value_counts()

In [ ]:
top_tissues = list(tissue_broad.value_counts()[tissue_broad.value_counts().sort_values()>150].index)

In [ ]:
tissue_bool_mask = (np.array([df_dedup.index.str.startswith(v) for v in top_tissues]).sum(axis=0)>0)

In [ ]:
df_module_cluster = df_icc_filt[tissue_bool_mask].sample(frac=1,random_state=12)

In [ ]:
df_module_cluster.shape

In [ ]:
X = StandardScaler().fit_transform(df_module_cluster)

n_pcs = 5
pca = PCA(n_components=n_pcs)
Z = pca.fit_transform(X)

tissue_fine = df_module_cluster.index.str.split('__').str[1]
tissue_broad = df_module_cluster.index.str.split('__').str[0]


In [ ]:
n_pcs_plot=5
xv = [v+1 for v in range(n_pcs_plot)]
plt.plot(xv,pca.explained_variance_ratio_[:n_pcs_plot],marker='o')
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
plt.xticks(xv)

figname = f'PCA_explainedVar_{n_pcs_plot}PCs_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tissue_broad.unique()

In [ ]:
tissue_cmaps = {
    # nervous / neuroendocrine (purples)
    "brain": "Purples", "nerve": "BuPu", "pituitary": "PuRd",
    # cardiovascular / blood (reds)
    "heart": "Reds", "blood_vessel": "OrRd", "blood": "YlOrRd", "spleen": "RdPu",
    # digestive (oranges / browns)
    "esophagus": "Oranges", "stomach": "YlOrBr", "small_intestine": "copper", "colon": "pink",
    "liver": "Greens", "pancreas": "YlGn",
    # endocrine (blue-greens)
    "thyroid": "BuGn", "adrenal_gland": "PuBuGn",
    # respiratory (blue)
    "lung": "Blues",
    # reproductive (blue-purples)
    "breast": "GnBu", "testis": "PuBu", "prostate": "YlGnBu",
    # structural (greys / neutrals)
    "skin": "bone", "muscle": "Greys", "adipose_tissue": "cividis",
}

In [ ]:
fine = tissue_fine
coarse = tissue_broad

group_cmaps = tissue_cmaps

fams = pd.Series(coarse.to_numpy(), index=fine.to_numpy()).groupby(level=0).first()
palette = {f: c for g, cm in group_cmaps.items()
           for f, c in zip(sorted(fams.index[fams == g]),
                           sns.color_palette(cm, (fams == g).sum() + 2)[2:])}
hue_order = sorted(palette, key=lambda f: (fams[f], f))

fams_c = pd.Series(coarse.to_numpy(), index=coarse.to_numpy()).groupby(level=0).first()
palette_c = {f: c for g, cm in group_cmaps.items()
           for f, c in zip(sorted(fams_c.index[fams_c == g]),
                           sns.color_palette(cm, (fams_c == g).sum() + 2)[2:])}
hue_order_c = sorted(palette_c, key=lambda f: (fams_c[f], f))


In [ ]:
n_pcs_plot=3
pc_names = [f"PC{i + 1}" for i in range(n_pcs_plot)]
Z_pl = Z[:,:n_pcs_plot]

pc_df = pd.DataFrame(Z_pl, columns=pc_names)
pc_df["fine"] = fine.to_numpy()
pc_df["coarse"] = coarse.to_numpy()

# --- pairplot ---

g = sns.PairGrid(pc_df, x_vars=pc_names[:-1], y_vars=pc_names[1:], hue="coarse",
                 hue_order=None, palette='tab10', height=2.2, corner=True)
g.map_offdiag(sns.scatterplot, s=8, alpha=0.7, linewidth=0)
g.add_legend()
sns.move_legend(g, "upper right", bbox_to_anchor=(0.98, 0.98), ncol=2,
                title=None, frameon=False, fontsize=8, markerscale=1.5)
g.legend.set_title(None)
plt.tight_layout()
figname = f'PCA_coarse_{n_pcs_plot}PCs_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.manifold import TSNE

X = StandardScaler().fit_transform(df_module_cluster)
n_pcs = 50
pca = PCA(n_components=n_pcs)
Z = pca.fit_transform(X)

E = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Z)

In [ ]:
plot_df = pd.DataFrame({"tSNE1": E[:, 0], "tSNE2": E[:, 1], "fine": fine.to_numpy(), "coarse":coarse.to_numpy()})

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=plot_df, x="tSNE1", y="tSNE2", hue="fine",
                hue_order=None,
                palette='tab10', s=12, alpha=0.8, linewidth=0)
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")

In [ ]:
tissue_broad.unique()